# 第 6 章 · 支持向量机 (SVM)

从零用 **hinge loss + 梯度下降** 实现线性二分类 SVM `MySVM`，与 sklearn `LinearSVC` 对比，应用到 sklearn 手写数字的 0 vs 1 二分类。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# 中文字体配置（仅在此cell设置一次，后续cell直接使用plt即可）
import os, platform
if platform.system() == 'Linux' and os.path.exists('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf'):
    fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11

COLORS = {
    'primary': '#6b6144', 'accent': '#96781b', 'accent2': '#459ebb',
    'pos': '#c0392b', 'neg': '#2980b9', 'green': '#3d8e58', 'muted': '#88867f'
}

import pandas as pd
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


## 6.1 算法原理

线性 SVM 求最大间隔超平面，等价于最小化 (L2 正则 + 合页损失)：

$$\min_{w,b}\ \frac{1}{2}\|w\|^2 + C\sum_i\max(0, 1-y_i(w^\top x_i+b))$$

对违反间隔 ($y_i f_i<1$) 的样本，梯度：

$$\nabla_w = w - C\sum_{violated} y_i x_i,\quad \nabla_b = -C\sum_{violated} y_i$$

用梯度下降迭代更新 $w,b$。

> PDF 伪代码步骤：① 初始化 ② 遍历样本 → 间隔满足/违反间隔 → 参数更新。

## 6.2 从零实现 (核心)

代码注释中的步骤编号与 PDF 伪代码一一对应。

In [ ]:
class MySVM:
    """从零实现的线性 SVM (hinge loss 梯度下降, 二分类)。"""
    def __init__(self, C=1.0, learning_rate=0.001, n_epochs=1000):
        self.C = C
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        # 步骤1: 初始化 w=0, b=0, 标签转±1
        self.classes_ = np.unique(y)
        y_bin = np.where(y == self.classes_[0], -1.0, 1.0)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0) + 1e-8
        Xs = (X - self.mean_) / self.std_
        n_samples, n_features = Xs.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        # 步骤2: 对每个样本判断间隔
        for _ in range(self.n_epochs):
            margins = y_bin * (Xs @ self.w + self.b)
            mask = margins < 1  # 违反间隔的样本
            mask_ok = ~mask     # 间隔满足的样本
            # 间隔满足: ∇_w = w（仅正则项）
            grad_w = self.w.copy()
            # 违反间隔: ∇_w = w - C·y_i·x_i, ∇_b = -C·y_i
            grad_w -= self.C * (y_bin[mask, None] * Xs[mask]).sum(axis=0)
            grad_b = -self.C * y_bin[mask].sum()
            # 参数更新 w ← w - α·∇_w
            self.w -= self.learning_rate * grad_w
            self.b -= self.learning_rate * grad_b
        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        Xs = (X - self.mean_) / self.std_
        return Xs @ self.w + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0,
                        self.classes_[1], self.classes_[0])


## 6.3 简单数据验证 + 决策边界

线性可分数据，对比自定义 SVM 与 sklearn 的间隔与准确率。

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=300, centers=2, cluster_std=1.2, random_state=3)
y = np.where(y==0, 0, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=3)

my_svm = MySVM(C=1.0, learning_rate=0.01, n_epochs=2000).fit(X_tr, y_tr)
sk_svm = LinearSVC(C=1.0, max_iter=5000, random_state=3).fit(X_tr, y_tr)
print('自定义  acc=%.4f' % accuracy_score(y_te, my_svm.predict(X_te)))
print('sklearn  acc=%.4f' % accuracy_score(y_te, sk_svm.predict(X_te)))

def plot_db(model, X, y, ax, title):
    x_min,x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min,y_max = X[:,1].min()-1, X[:,1].max()+1
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,200), np.linspace(y_min,y_max,200))
    Z = np.asarray(model.predict(np.c_[xx.ravel(),yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.2,cmap=plt.cm.coolwarm)
    ax.scatter(X[:,0],X[:,1],c=y,cmap=plt.cm.coolwarm,edgecolor='k',s=30)
    ax.set_title(title)

fig, axes = plt.subplots(1,2,figsize=(11,4.5))
plot_db(my_svm, X, y, axes[0], '自定义 SVM')
plot_db(sk_svm, X, y, axes[1], 'sklearn LinearSVC')
plt.tight_layout(); plt.show()


## 6.4 与 sklearn 对比 (月牙数据, 4 种核函数)

线性 SVM 处理不了非线性可分数据。这里展示 4 种核函数：自定义线性 SVM vs sklearn 的 linear / poly / rbf / sigmoid 核，体会【核技巧】的威力。

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=400, noise=0.2, random_state=9)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=9)

my_svm = MySVM(C=1.0, learning_rate=0.01, n_epochs=2000).fit(X_tr, y_tr)
print('自定义(线性)  acc=%.4f' % accuracy_score(y_te, my_svm.predict(X_te)))

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
fig, axes = plt.subplots(2, 2, figsize=(11,9))
for ax, k in zip(axes.ravel(), kernels):
    sk = SVC(C=1.0, kernel=k, gamma='scale').fit(X_tr, y_tr)
    acc = accuracy_score(y_te, sk.predict(X_te))
    print(f'sklearn({k:<8}) acc={acc:.4f}')
    plot_db(sk, X, y, ax, f'SVC kernel={k} (acc={acc:.3f})')
plt.tight_layout(); plt.show()


## 6.5 真实应用 · 手写数字 0 vs 1

sklearn `load_digits` 是 8×8 手写数字，1797 张。这里做二分类：识别数字 0 还是 1。

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
mask = np.isin(digits.target, [0, 1])
X = digits.data[mask]
y = digits.target[mask]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

my_svm = MySVM(C=1.0, learning_rate=0.05, n_epochs=2000).fit(X_tr_s, y_tr)
sk_svm = LinearSVC(C=1.0, max_iter=10000, random_state=42).fit(X_tr_s, y_tr)
print('自定义  acc=%.4f' % accuracy_score(y_te, my_svm.predict(X_te_s)))
print('sklearn  acc=%.4f' % accuracy_score(y_te, sk_svm.predict(X_te_s)))
print(classification_report(y_te, my_svm.predict(X_te_s), digits=3))

cm = confusion_matrix(y_te, my_svm.predict(X_te_s))
fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(cm, cmap='YlOrBr')
for i in range(2):
    for j in range(2):
        ax.text(j,i,cm[i,j],ha='center',va='center')
ax.set_xticks([0,1]); ax.set_xticklabels(['0','1'])
ax.set_yticks([0,1]); ax.set_yticklabels(['0','1'])
ax.set_xlabel('预测'); ax.set_ylabel('真实'); plt.tight_layout(); plt.show()


## 6.6 小结

- 线性 SVM 用 hinge loss + 梯度下降即可手写，效果接近 sklearn `LinearSVC`；
- 非线性问题需要核技巧 (sklearn `SVC(kernel='rbf')`)，从零实现核 SVM 较复杂故对比；
- 手写数字 0/1 这种【高维线性可分】任务，线性 SVM 几乎完美。